In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset
file_path = "F:\\school\\Azubi Africa\\LP1 Data Analytics Project\\LP-1-Project\\data\\Aba3_cleaned.csv"
df = pd.read_csv(file_path)

# Data Cleaning and Preparation (Handling missing values and data types)
df['Founded'] = pd.to_datetime(df['Founded'], format='%Y', errors='coerce').dt.year # Extract year
df['Amount'] = pd.to_numeric(df['Amount',], errors='coerce') #changing to numeric.
df.dropna(subset=['Amount'], inplace=True) # Remove rows with missing funding amounts
df.fillna(0, inplace=True) # Fill other missing values with 0 (or a more appropriate value)
df = df[(df['Founded'] >= 1980) & (df['Founded'] <= 2021)]


In [ ]:
from scipy import stats
import statsmodels.formula.api as sm

#Remove outliers
Q1 = df['Amount'].quantile(0.25)
Q3 = df['Amount'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df_no_outliers = df[(df['Amount'] >= lower_bound) & (df['Amount'] <= upper_bound)]

# Perform ANOVA test
formula = 'Amount ~ C(Industry)'
lm = sm.ols(formula, data=df_no_outliers).fit()
anova_table = sm.stats.anova_lm(lm, typ=2)

print(anova_table)

# Interpretation:
# Look at the p-value in the ANOVA table.
# - If the p-value is less than your significance level (e.g., 0.05), you reject the null hypothesis. This suggests that there are statistically significant differences in average funding amounts across different industries.
# - If the p-value is greater than your significance level, you fail to reject the null hypothesis.  This suggests that there isn't enough evidence to conclude that the average funding amounts differ significantly across industries.


In [ ]:
# Calculate average funding per industry
avg_funding_per_industry = df.groupby('Industry')['Amount'].mean().sort_values(ascending=False)

print("Top 10 Industries by Average Funding:")
print(avg_funding_per_industry.head(10))

# Visualization (optional)
avg_funding_per_industry.head(10).plot(kind='bar', figsize=(10, 6))
plt.title('Top 10 Industries by Average Funding (1980-2021)')
plt.xlabel('Industry')
plt.ylabel('Average Funding Amount')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
# Calculate median funding amount for each round
median_funding_per_round = df.groupby('RoundSeries')['Amount'].median().sort_values(ascending=False)

print("\nMedian Funding by Round Series:")
print(median_funding_per_round)

# Visualization
median_funding_per_round.plot(kind='bar', figsize=(10, 6))
plt.title('Median Funding Amount by Round Series')
plt.xlabel('Funding Round')
plt.ylabel('Median Funding Amount')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Interpretation:
# The higher the round series the higher the funding amount


In [ ]:
# Filter data for the relevant years
funding_2018 = df[df['Founded'] == 2018]['Amount']
funding_2019 = df[df['Founded'] == 2019]['Amount']
funding_2020 = df[df['Founded'] == 2020]['Amount']

# Calculate summary statistics (mean, median)
print("\nFunding in 2018:")
print(f"  Mean: {funding_2018.mean():.2f}, Median: {funding_2018.median():.2f}")
print("\nFunding in 2019:")
print(f"  Mean: {funding_2019.mean():.2f}, Median: {funding_2019.median():.2f}")
print("\nFunding in 2020:")
print(f"  Mean: {funding_2020.mean():.2f}, Median: {funding_2020.median():.2f}")

# Perform statistical test (e.g., t-test or Mann-Whitney U test if data is not normally distributed)
# Example: Mann-Whitney U test for 2019 vs. 2020
from scipy.stats import mannwhitneyu
stat, p = mannwhitneyu(funding_2019, funding_2020)
print(f"\nMann-Whitney U test (2019 vs 2020): statistic={stat:.3f}, p-value={p:.3f}")

# Interpretation:
# Compare the mean and median funding amounts for each year.  Also, examine the p-value from the Mann-Whitney U test (or t-test). A p-value less than 0.05 suggests a statistically significant difference in funding amounts between the years.


In [ ]:
# Calculate average funding per industry for pre-pandemic (2018-2019)
pre_pandemic = df[(df['Founded'] == 2018) | (df['Founded'] == 2019)].groupby('Industry')['Amount'].mean()

# Calculate average funding per industry during the pandemic (2020)
pandemic = df[df['Founded'] == 2020].groupby('Industry')['Amount'].mean()

# Calculate the percentage change in funding
funding_change = ((pandemic - pre_pandemic) / pre_pandemic) * 100
funding_change = funding_change.sort_values(ascending=False)

print("\nIndustries with Significant Funding Changes (Pre-Pandemic vs. Pandemic):")
print(funding_change)

# Identify thriving and struggling sectors
thriving_sectors = funding_change[funding_change > 0].index.tolist()
struggling_sectors = funding_change[funding_change < 0].index.tolist()

print("\nThriving Sectors:", thriving_sectors)
print("Struggling Sectors:", struggling_sectors)


In [ ]:
# Calculate average funding per location
avg_funding_per_location = df.groupby('Head_Quarter')['Amount'].mean().sort_values(ascending=False)

print("\nLocations with Highest Average Funding:")
print(avg_funding_per_location)

# You can further analyze this by combining location with other factors (e.g., industry)


In [ ]:
# Create a pivot table
pivot_table = df.pivot_table(values='Amount', index='Head_Quarter', columns='Industry', aggfunc='sum', fill_value=0)

# Create the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_table, cmap="YlGnBu", linewidths=.5)
plt.title('Heatmap of Total Funding Amount by Location and Industry')
plt.xlabel('Industry')
plt.ylabel('Location')
plt.tight_layout()
plt.show()


In [ ]:
# Filter the data for the years 2010 to 2021
df_2010_2021 = df[(df['Founded'] >= 2010) & (df['Founded'] <= 2021)]

# Group by year and calculate the total funding amount for each year
yearly_funding = df_2010_2021.groupby('Founded')['Amount'].sum()

# Plot the yearly funding trends
plt.figure(figsize=(12, 6))
yearly_funding.plot(kind='line', marker='o')
plt.title('Total Funding Amount for Indian Start-ups (2010-2021)')
plt.xlabel('Year')
plt.ylabel('Total Funding Amount')
plt.grid(True)
plt.show()

# Calculate the percentage change in funding from 2019 to 2020
funding_2019 = yearly_funding.loc[2019] if 2019 in yearly_funding.index else 0
funding_2020 = yearly_funding.loc[2020] if 2020 in yearly_funding.index else 0

if funding_2019 != 0:
    percentage_change = ((funding_2020 - funding_2019) / funding_2019) * 100
    print(f"Percentage change in funding from 2019 to 2020: {percentage_change:.2f}%")
else:
    print("Funding data for 2019 is not available.")
